# Aula Prática — Semânticas de Entrega no Apache Kafka
### Computação Paralela e Distribuída (DCE540)

Nesta aula vamos aprofundar o que já vimos sobre Kafka, respondendo a uma pergunta central para qualquer sistema distribuído de mensageria:

> **Se um consumidor falhar no meio do processamento, o que acontece com a mensagem que ele estava tratando? Ela é perdida? É processada de novo? Nenhuma das duas?**

Vamos estudar e demonstrar, na prática, as três respostas possíveis que o Kafka permite configurar:

1. **At-most-once** (no máximo uma vez) — mensagens podem ser perdidas, nunca duplicadas;
2. **At-least-once** (pelo menos uma vez) — mensagens nunca são perdidas, mas podem ser duplicadas;
3. **Exactly-once** (exatamente uma vez) — mensagens nunca são perdidas nem duplicadas.

Ao final, três atividades práticas consolidam os três conceitos.

**Pré-requisito:** aula anterior de introdução ao Kafka (tópicos, partições, offset, producer, consumer, consumer group).


## 0. Preparando o ambiente (VS Code + Docker)

Esta aula usa um ambiente Kafka **novo e independente** da aula anterior (pasta própria, com seu próprio `docker-compose.yml`). Siga estes passos:

1. Confirme que o **VS Code** tem as extensões **Python** e **Jupyter** instaladas (se já configurou na aula 1, não precisa repetir).
2. Confirme que o **Docker Desktop** está aberto e rodando (ícone da baleia estável na barra de tarefas).
3. **Importante:** se os containers da aula anterior (`kafka-aula`, `kafka-ui-aula`) ainda estiverem rodando, pare-os primeiro — esta aula usa as mesmas portas (9092, 9094, 8080) e haverá conflito:

   ```powershell
   # na pasta da AULA 1, se ela ainda estiver de pé
   docker compose down
   ```

4. Abra a pasta **desta aula** no VS Code (File > Open Folder) — todos os arquivos (`docker-compose.yml`, este notebook, `requirements.txt`) devem estar juntos na mesma pasta.
5. Em um terminal do VS Code (**Terminal > New Terminal**), crie o ambiente virtual e instale as dependências:

   **Windows (PowerShell):**
   ```powershell
   python -m venv venv
   venv\Scripts\activate
   python -m pip install --upgrade pip
   python -m pip install -r requirements.txt
   ```

   **Linux/Mac:**
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   python3 -m pip install --upgrade pip
   python3 -m pip install -r requirements.txt
   ```

   > **Use sempre `python -m pip install ...`, nunca `pip install ...` sozinho.** Em pastas dentro do OneDrive/Google Drive, o `pip.exe` isolado pode acabar sendo o do Python global do sistema em vez do `venv` — mesmo com o prompt mostrando `(venv)` — e os pacotes vão parar no lugar errado sem erro nenhum, só um aviso fácil de passar despercebido (`Defaulting to user installation...`).

   > No Windows, o comando costuma ser `python` (não `python3`). Se der erro dizendo que o Python não foi encontrado, rode `python --version` primeiro para confirmar que ele está instalado e acessível nesse terminal.

6. Abra `aula2_semanticas_entrega.ipynb` no VS Code e selecione o kernel `venv` (canto superior direito do notebook).
7. No terminal, suba o ambiente Kafka desta aula:

   ```bash
   docker compose up -d
   ```

   Aguarde ~15-20 segundos para o broker inicializar. Confira com `docker compose ps` — os containers `kafka-aula2` e `kafka-ui-aula2` devem aparecer com status `running`.

8. (Opcional) Acompanhe visualmente em `http://localhost:8080` (Kafka UI).

A partir daqui, execute as células em ordem (`Shift+Enter`). Ao final da aula, encerre o ambiente:

```bash
docker compose down -v
```


## 1. Contextualização e o problema motivador

### Relembrando rapidamente

Um **consumer** lê mensagens de um tópico e periodicamente informa ao Kafka até qual **offset** (posição) ele já processou — isso se chama **commit**. Se o consumidor cair e for reiniciado (sozinho ou substituído por outro do mesmo **consumer group**), ele retoma a leitura a partir do último offset commitado, não do início do tópico.

O detalhe crucial, que normalmente passa despercebido, é: **o commit do offset e o processamento da mensagem são duas operações separadas**, e a ordem/atomicidade entre elas é o que define qual garantia de entrega você tem.

### O cenário motivador

Imagine um consumidor que lê pedidos de um tópico e soma seus valores em um totalizador. Para cada mensagem, ele precisa fazer duas coisas:

- **(P)** Processar a mensagem (ex.: somar o valor ao totalizador);
- **(C)** Commitar o offset (avisar ao Kafka "já tratei até aqui").

Se o processo cair exatamente **entre** essas duas operações, o que aconteceu antes da queda determina o resultado ao reiniciar. Existem três formas de organizar P e C:

| Ordem das operações | O que acontece se cair entre P e C | Garantia resultante |
|---|---|---|
| **C antes de P** (commita, depois processa) | Ao reiniciar, o Kafka acha que a mensagem já foi tratada (offset já avançou) — mas ela nunca chegou a ser processada de fato. **Mensagem perdida.** | **At-most-once** |
| **P antes de C** (processa, depois commita) | Ao reiniciar, o offset ainda não avançou — a mesma mensagem é lida e processada **de novo**. **Mensagem duplicada.** | **At-least-once** |
| **P e C acontecem como uma única operação atômica** (tudo ou nada) | Não existe "meio do caminho" — ou os dois aconteceram, ou nenhum. | **Exactly-once** |

Nenhuma escolha é "a errada" — cada uma tem seu custo e seu caso de uso, como veremos.

### Preparando o ambiente do exercício

Vamos usar um tópico `pedidos` com **1 única partição** (propositalmente — isso evita que a distribuição entre partições, já estudada na aula anterior, "misture" com o que queremos observar aqui: o comportamento do offset dentro de uma única sequência de mensagens).


In [14]:
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient, TopicPartition
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
import json
import time
import uuid

BOOTSTRAP_SERVERS = "localhost:9094"
TOPIC = "pedidos"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=1, replication_factor=1)])
    print(f"Tópico '{TOPIC}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC}' já existia.")
admin.close()


Tópico 'pedidos' já existia.


Agora, uma função auxiliar que publica `n` pedidos no tópico. Cada pedido tem um **identificador único** (`id`) — isso não é usado ainda, mas será importante na Atividade 2, quando formos implementar um consumidor idempotente.


In [15]:
def publicar_pedidos(n, valor_fixo=10.0):
    """Publica n pedidos sequenciais no tópico, cada um com um id único e o mesmo valor
    (valor fixo facilita conferir se o total bateu ao final: total esperado = n * valor_fixo)."""
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )
    for i in range(n):
        pedido = {"id": str(uuid.uuid4()), "sequencia": i, "valor": valor_fixo}
        producer.send(TOPIC, value=pedido)
    producer.flush()
    producer.close()
    print(f"{n} pedidos publicados (valor total esperado: {n * valor_fixo}).")

publicar_pedidos(10)


10 pedidos publicados (valor total esperado: 100.0).


/tmp/ipykernel_30852/2927629279.py:15: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  publicar_pedidos(10)


## 2. At-most-once (no máximo uma vez)

**Configuração:** `enable_auto_commit=False` (controle manual) + **commit do offset ANTES de processar** a mensagem.

Essa ordem parece contraintuitiva à primeira vista ("por que eu ia commitar antes de processar?"), mas é exatamente o que acontece, na prática, quando se usa o **auto-commit padrão do Kafka** (`enable_auto_commit=True`): o commit acontece em um intervalo de tempo fixo (`auto_commit_interval_ms`), **independente** de o processamento da mensagem ter terminado com sucesso ou não. Se o processamento falhar depois que o auto-commit já rodou, a mensagem é dada como "tratada" mesmo sem ter sido.

Vamos simular isso de forma controlada: um consumidor que **comita o offset assim que recebe a mensagem**, e só depois tenta processá-la (somar ao total). No meio do caminho, vamos forçar uma falha proposital na mensagem de sequência `FALHA_NA_SEQUENCIA`, simulando uma queda do processo exatamente nesse ponto.


In [16]:
FALHA_NA_SEQUENCIA = 5  # a mensagem de índice 5 (a 6ª) vai "derrubar" o consumidor

def consumir_at_most_once(group_id, falhar_em=None, estado=None):
    # 'estado' é um dicionário mutável usado para guardar o progresso.
    # Isso garante que, mesmo se a função for interrompida por uma exceção
    # (simulando uma queda), o chamador ainda consegue inspecionar o que
    # já tinha sido processado até aquele ponto.
    if estado is None:
        estado = {"total": 0.0, "processados": []}

    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        enable_auto_commit=False,          # controle manual do commit
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )

    try:
        for msg in consumer:
            pedido = msg.value

            # (C) Commit ANTES de processar — é isso que caracteriza at-most-once
            consumer.commit()

            if falhar_em is not None and pedido["sequencia"] == falhar_em:
                print(f"Total acumulado até aqui (antes da falha): {estado['total']}")
                print(f"!! Simulando falha ao tentar processar a mensagem de sequência {pedido['sequencia']} !!")
                raise RuntimeError("Falha simulada no consumidor")

            # (P) Processamento (só chega aqui se o commit acima já ocorreu)
            estado["total"] += pedido["valor"]
            estado["processados"].append(pedido["sequencia"])

    finally:
        consumer.close()

    return estado["total"], estado["processados"]

# Primeira execução: vai cair na mensagem de sequência 5, DEPOIS de já ter commitado o offset dela
estado_execucao_1 = {"total": 0.0, "processados": []}
try:
    total, processados = consumir_at_most_once("grupo-at-most-once", falhar_em=FALHA_NA_SEQUENCIA, estado=estado_execucao_1)
except RuntimeError as e:
    print(f"Consumidor encerrado por falha: {e}")
    print(f"Sequências processadas antes da falha: {estado_execucao_1['processados']}")


/tmp/ipykernel_30852/1392699802.py:11: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Total acumulado até aqui (antes da falha): 50.0
!! Simulando falha ao tentar processar a mensagem de sequência 5 !!
Consumidor encerrado por falha: Falha simulada no consumidor
Sequências processadas antes da falha: [0, 1, 2, 3, 4]


Repare que a mensagem de sequência 5 **nunca chegou a ser somada** ao total (a exceção interrompeu a execução antes da linha `estado["total"] += pedido["valor"]`), mas o `consumer.commit()` **já tinha sido chamado** para ela.

Agora vamos "reiniciar" o consumidor (mesmo `group_id`, sem forçar nova falha) e ver o que acontece:


In [17]:
# "Reiniciando" o consumidor — mesmo group_id, sem falha desta vez
total_final, processados = consumir_at_most_once("grupo-at-most-once", falhar_em=None)

print(f"Sequências processadas após o reinício: {processados}")
print(f"Total acumulado (só nesta execução): {total_final}")
print(f"Valor esperado originalmente (10 pedidos x 10.0): 100.0")
print(f"A mensagem de sequência {FALHA_NA_SEQUENCIA} foi perdida — o offset dela já tinha sido commitado antes da falha.")


/tmp/ipykernel_30852/1392699802.py:11: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Sequências processadas após o reinício: [6, 7, 8, 9, 10, 11, 12, 13, 14, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total acumulado (só nesta execução): 340.0
Valor esperado originalmente (10 pedidos x 10.0): 100.0
A mensagem de sequência 5 foi perdida — o offset dela já tinha sido commitado antes da falha.


## 3. At-least-once (pelo menos uma vez)

**Configuração:** `enable_auto_commit=False` + **commit do offset DEPOIS de processar** a mensagem com sucesso.

Essa é a garantia mais comum em aplicações reais, porque é a mais fácil de obter "com segurança" (nunca perder dados) — o preço é que a aplicação precisa estar preparada para lidar com mensagens repetidas.

Vamos repetir o mesmo experimento, mas invertendo a ordem: processar primeiro, commitar depois.


In [18]:
def consumir_at_least_once(group_id, falhar_em=None, estado=None):
    if estado is None:
        estado = {"total": 0.0, "processados": []}

    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        enable_auto_commit=False,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )

    try:
        for msg in consumer:
            pedido = msg.value

            # (P) Processamento ACONTECE ANTES do commit
            estado["total"] += pedido["valor"]
            estado["processados"].append(pedido["sequencia"])

            if falhar_em is not None and pedido["sequencia"] == falhar_em:
                print(f"Total acumulado até aqui (antes da falha): {estado['total']}")
                print(f"!! Simulando falha DEPOIS de processar a mensagem de sequência {pedido['sequencia']}, ANTES do commit !!")
                raise RuntimeError("Falha simulada no consumidor")

            # (C) Commit só depois que o processamento terminou
            consumer.commit()

    finally:
        consumer.close()

    return estado["total"], estado["processados"]

# Reaproveitamos o MESMO tópico "pedidos" e os mesmos 10 pedidos publicados na Seção 1 —
# não é preciso publicar de novo. Como o group_id abaixo é novo ("grupo-at-least-once"),
# ele nunca leu nada deste tópico antes, então recomeça do início (auto_offset_reset="earliest").
estado_execucao_1 = {"total": 0.0, "processados": []}
try:
    total, processados = consumir_at_least_once("grupo-at-least-once", falhar_em=FALHA_NA_SEQUENCIA, estado=estado_execucao_1)
except RuntimeError as e:
    print(f"Consumidor encerrado por falha: {e}")
    print(f"Sequências processadas antes da falha: {estado_execucao_1['processados']}")


Total acumulado até aqui (antes da falha): 60.0
!! Simulando falha DEPOIS de processar a mensagem de sequência 5, ANTES do commit !!
Consumidor encerrado por falha: Falha simulada no consumidor
Sequências processadas antes da falha: [0, 1, 2, 3, 4, 5]


/tmp/ipykernel_30852/42309962.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


In [19]:
# "Reiniciando" o consumidor — o offset NÃO avançou além da mensagem que falhou,
# então ela (e as seguintes) serão lidas de novo.
total_final, processados = consumir_at_least_once("grupo-at-least-once", falhar_em=None)

print(f"Sequências processadas após o reinício: {processados}")
print(f"(Repare que a sequência {FALHA_NA_SEQUENCIA} aparece de novo — foi reprocessada.)")


/tmp/ipykernel_30852/42309962.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Sequências processadas após o reinício: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
(Repare que a sequência 5 aparece de novo — foi reprocessada.)


Se você somar o total das duas execuções, o valor da mensagem de sequência 5 foi contado **duas vezes**. Nenhuma mensagem foi perdida — mas o totalizador ficou incorreto por causa da duplicidade.

### Por que isso importa: idempotência

Para usar at-least-once com segurança, a aplicação que consome as mensagens precisa ser **idempotente**: processar a mesma mensagem duas (ou mais) vezes deve ter o **mesmo efeito** que processá-la uma vez só.

Alguns exemplos de como conseguir isso:
- Em vez de `saldo += valor` (não-idempotente: aplicar duas vezes soma duas vezes), usar `saldo = valor_final_calculado` (idempotente: aplicar várias vezes dá o mesmo resultado);
- Manter um registro (tabela, conjunto, cache) de **ids de mensagens já processadas**, e pular qualquer id repetido antes de aplicar o efeito;
- Usar operações que o próprio destino já trata como idempotentes (ex.: `INSERT ... ON CONFLICT DO NOTHING` em um banco de dados, usando o `id` do pedido como chave).

É exatamente essa segunda estratégia que a Atividade 2 vai pedir para implementar.


## 4. Exactly-once (exatamente uma vez)

Exactly-once não é uma configuração mágica única — é a combinação de **duas garantias independentes**:

### 4.1 Producer idempotente (evita duplicação na *escrita*)

Mesmo um único producer pode gerar duplicatas: se ele envia uma mensagem e a confirmação (*ack*) do broker se perde na rede (não o dado, só a confirmação), o producer, sem saber se a mensagem chegou, tenta reenviar — e agora o broker recebeu a mesma mensagem duas vezes.

Um **producer idempotente** resolve isso: cada producer recebe um ID interno e numera sequencialmente cada mensagem que envia. O broker usa esse par (ID do producer, número de sequência) para **descartar duplicatas** automaticamente, mesmo em reenvios.


In [20]:
# A partir da versão do kafka-python usada nesta aula, enable_idempotence já vem True por padrão,
# mas vamos deixar explícito para fins didáticos.
producer_idempotente = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    enable_idempotence=True,   # ativa numeração de sequência + deduplicação no broker
    acks="all",                # exigido para idempotência funcionar corretamente
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
print("Producer idempotente configurado. (enable_idempotence evita duplicar mensagens em reenvios de rede.)")
producer_idempotente.close()


Producer idempotente configurado. (enable_idempotence evita duplicar mensagens em reenvios de rede.)


Isso resolve duplicação **na escrita**, mas sozinho não resolve o problema do consumidor que vimos nas seções 2 e 3 — para isso, precisamos de **transações**.

### 4.2 Transações (acopla processar + commitar como uma única operação atômica)

Uma transação no Kafka permite agrupar múltiplas operações (produzir mensagens, e até commitar offsets de um consumo) em um único bloco que **ou acontece por completo, ou não acontece nada** — exatamente a "operação atômica" que faltava na nossa tabela da Seção 1.

Vamos demonstrar o comportamento básico: iniciar uma transação, produzir mensagens, e comparar o que acontece quando ela é **commitada** versus **abortada**.

**Detalhe técnico importante:** o `producer.send()` não envia a mensagem imediatamente — ele só a coloca em um buffer interno, que é transmitido ao broker em lotes (por eficiência). Por isso, antes de `commit_transaction()` ou `abort_transaction()`, chamamos `producer.flush()` explicitamente, para garantir que as mensagens realmente saiam do buffer e cheguem ao broker como parte da transação. Sem isso, mensagens ainda não transmitidas no momento do abort podem ser simplesmente descartadas antes de nunca chegarem a existir no log do Kafka — o que impediria a próxima demonstração (leitura com `read_uncommitted`) de mostrar qualquer diferença.


In [21]:
TOPIC_TRANSACIONAL = "pedidos-transacional"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC_TRANSACIONAL, num_partitions=1, replication_factor=1)])
    print(f"Tópico '{TOPIC_TRANSACIONAL}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC_TRANSACIONAL}' já existia.")
admin.close()

producer_tx = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    transactional_id="produtor-pedidos-tx-1",   # identifica esta transação de forma única e estável
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

producer_tx.init_transactions()   # inicializa o coordenador de transações para este producer

# --- Transação 1: será COMMITADA ---
producer_tx.begin_transaction()
for i in range(3):
    producer_tx.send(TOPIC_TRANSACIONAL, value={"sequencia": i, "lote": "commitado"})
producer_tx.flush()               # garante que as mensagens realmente saíram para o broker
producer_tx.commit_transaction()
print("Transação 1 (3 mensagens) commitada com sucesso.")

# --- Transação 2: será ABORTADA (simulando uma falha detectada antes do commit) ---
producer_tx.begin_transaction()
for i in range(3, 6):
    producer_tx.send(TOPIC_TRANSACIONAL, value={"sequencia": i, "lote": "abortado"})
producer_tx.flush()               # IMPORTANTE: sem isso, as mensagens podem nunca chegar
                                   # a ser transmitidas ao broker antes do abort — e nesse
                                   # caso o read_uncommitted não teria nada "sujo" para mostrar.
producer_tx.abort_transaction()
print("Transação 2 (3 mensagens) ABORTADA propositalmente.")

producer_tx.close()


Tópico 'pedidos-transacional' já existia.
Transação 1 (3 mensagens) commitada com sucesso.
Transação 2 (3 mensagens) ABORTADA propositalmente.


Agora vamos ler o tópico com dois consumidores diferentes, variando apenas o parâmetro `isolation_level`:

- `read_uncommitted` (padrão do Kafka) — enxerga **todas** as mensagens, mesmo as de transações abortadas;
- `read_committed` — só enxerga mensagens de transações que foram **commitadas** com sucesso.


In [22]:
def ler_topico_transacional(isolation_level):
    consumer = KafkaConsumer(
        TOPIC_TRANSACIONAL,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=f"grupo-leitura-{isolation_level}",
        auto_offset_reset="earliest",
        isolation_level=isolation_level,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    mensagens = [msg.value for msg in consumer]
    consumer.close()
    return mensagens

print("read_uncommitted:", ler_topico_transacional("read_uncommitted"))
print()
print("read_committed:  ", ler_topico_transacional("read_committed"))


/tmp/ipykernel_30852/4225631569.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


read_uncommitted: [{'sequencia': 0, 'lote': 'commitado'}, {'sequencia': 1, 'lote': 'commitado'}, {'sequencia': 2, 'lote': 'commitado'}, {'sequencia': 3, 'lote': 'abortado'}, {'sequencia': 4, 'lote': 'abortado'}, {'sequencia': 5, 'lote': 'abortado'}]

read_committed:   [{'sequencia': 0, 'lote': 'commitado'}, {'sequencia': 1, 'lote': 'commitado'}, {'sequencia': 2, 'lote': 'commitado'}]


Com `read_committed`, as mensagens do "lote abortado" simplesmente **não existem** do ponto de vista do consumidor — como se a transação inteira nunca tivesse acontecido. É esse comportamento tudo-ou-nada que permite construir pipelines exactly-once: se o processamento de um lote falhar antes do commit da transação, basta abortar, e nenhum efeito parcial fica visível para quem lê os dados a seguir.

**Importante para perspectiva:** isso garante exactly-once **dentro do ecossistema Kafka** (entre um tópico de entrada e um de saída, por exemplo, no padrão *consume-transform-produce*). Se o destino final for um sistema externo (um banco de dados, uma API), garantir exactly-once "de ponta a ponta" exige mecanismos adicionais nesse sistema externo — o Kafka sozinho não resolve isso além de sua própria fronteira.


## 5. Atividades Práticas

As três atividades a seguir usam o que foi demonstrado nas seções anteriores, mas pedem para você **escrever** a solução, não apenas executar código pronto.


### Atividade 1 — Alterando o ponto de falha (at-most-once vs. at-least-once)

**Objetivo:** confirmar, na prática, que a garantia de entrega depende exclusivamente da ordem entre processar e commitar — não da mensagem em si.

**Tarefa:**
1. Publique 15 novos pedidos com `publicar_pedidos(15)`.
2. Copie a função `consumir_at_least_once` para uma nova função `consumir_configuravel`, adicionando um parâmetro `commit_antes_de_processar: bool`.
   - Se `True`, deve se comportar como at-most-once (commit antes do processamento);
   - Se `False`, deve se comportar como at-least-once (commit depois).
3. Rode o experimento **duas vezes** (uma para cada valor do parâmetro), sempre com um `group_id` novo e `falhar_em` apontando para alguma mensagem no meio do lote.
4. Para cada execução, calcule o total processado ao final (some as duas rodadas: antes e depois do "reinício") e compare com o valor esperado (`15 * valor_fixo`).

**Pergunta para o relatório:** os totais confirmam a teoria (um ficou menor, o outro maior, que o valor esperado)? Em qual dos dois cenários você prefereria estar se o "pedido" fosse, na vida real, uma cobrança de cartão de crédito? E se fosse uma leitura de sensor de temperatura a cada segundo?


In [23]:
# Espaço para a Atividade 1

def obter_offset_final(topico):
    """Retorna o offset em que o próximo lote começará."""
    consumer = KafkaConsumer(bootstrap_servers=BOOTSTRAP_SERVERS, enable_auto_commit=False)
    tp = TopicPartition(topico, 0)
    consumer.assign([tp])
    offset = consumer.end_offsets([tp])[tp]
    consumer.close()
    return offset

def consumir_configuravel(group_id, commit_antes_de_processar, falhar_em=None,
                          estado=None, offset_inicial=None):
    if estado is None:
        estado = {"total": 0.0, "processados": []}

    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        enable_auto_commit=False,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    tp = TopicPartition(TOPIC, 0)
    consumer.assign([tp])
    if offset_inicial is not None:
        consumer.seek(tp, offset_inicial)

    try:
        for msg in consumer:
            pedido = msg.value

            if commit_antes_de_processar:
                consumer.commit()  # at-most-once: C antes de P

            if falhar_em is not None and pedido["sequencia"] == falhar_em:
                if not commit_antes_de_processar:
                    # Em at-least-once, a falha ocorre depois de P e antes de C.
                    estado["total"] += pedido["valor"]
                    estado["processados"].append(pedido["sequencia"])
                raise RuntimeError(f"Falha simulada na sequência {falhar_em}")

            estado["total"] += pedido["valor"]
            estado["processados"].append(pedido["sequencia"])

            if not commit_antes_de_processar:
                consumer.commit()  # at-least-once: C depois de P
    finally:
        consumer.close()

    return estado["total"], estado["processados"]

# Marcamos o início do lote para ignorar os 10 pedidos usados nas demonstrações anteriores.
inicio_lote_atividade1 = obter_offset_final(TOPIC)
publicar_pedidos(15)
falha_atividade1 = 7
resultados_atividade1 = {}

for commit_antes, nome in [(True, "at-most-once"), (False, "at-least-once")]:
    estado = {"total": 0.0, "processados": []}
    grupo = f"atividade1-{nome}-{uuid.uuid4()}"
    try:
        consumir_configuravel(
            grupo, commit_antes, falha_atividade1, estado, inicio_lote_atividade1
        )
    except RuntimeError as erro:
        print(f"{nome}: {erro}; reiniciando o consumidor...")

    # Mesmo grupo: retoma do offset que ficou commitado.
    consumir_configuravel(grupo, commit_antes, estado=estado)
    resultados_atividade1[nome] = estado["total"]
    print(f"{nome}: total = {estado['total']:.1f}; sequências = {estado['processados']}")

esperado_atividade1 = 15 * 10.0
print(f"Esperado sem perda ou duplicidade: {esperado_atividade1:.1f}")


/tmp/ipykernel_30852/595291747.py:56: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  publicar_pedidos(15)
/tmp/ipykernel_30852/595291747.py:17: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


15 pedidos publicados (valor total esperado: 150.0).
at-most-once: Falha simulada na sequência 7; reiniciando o consumidor...
at-most-once: total = 140.0; sequências = [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14]
at-least-once: Falha simulada na sequência 7; reiniciando o consumidor...
at-least-once: total = 160.0; sequências = [0, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, 13, 14]
Esperado sem perda ou duplicidade: 150.0


**Resposta da Atividade 1.** Sim. Com falha na sequência 7, o cenário *at-most-once* totaliza **140,0**, pois a mensagem da falha é commitada e depois perdida. O *at-least-once* totaliza **160,0**, pois essa mensagem é processada antes da falha e novamente após o reinício. O valor sem anomalias seria **150,0**. Para cobranças de cartão, entre essas duas opções isoladas eu preferiria *at-most-once*, porque uma cobrança duplicada é especialmente danosa; em produção, porém, o ideal seria *at-least-once* com idempotência. Para leituras de temperatura a cada segundo, *at-most-once* costuma ser aceitável, pois perder uma amostra geralmente é menos grave que duplicá-la e enviesar as análises.


### Atividade 2 — Consumidor idempotente (tornando o at-least-once seguro)

**Objetivo:** implementar a solução de idempotência discutida na Seção 3, eliminando o efeito da duplicação sem abrir mão da garantia de "nunca perder mensagens".

**Tarefa:**
1. Publique 15 novos pedidos com `publicar_pedidos(15)` (cada um já tem um `id` único, gerado com `uuid.uuid4()`).
2. Escreva uma função `consumir_idempotente(group_id, falhar_em=None)`, baseada em `consumir_at_least_once` (processa antes, commita depois — ou seja, ainda é at-least-once "por baixo dos panos").
3. Dentro da função, mantenha um `set()` (em memória, para fins didáticos) chamado `ids_processados`.
4. Antes de somar o valor de um pedido ao total, verifique se o `id` dele já está em `ids_processados`:
   - Se **já estiver**, pule a soma (mas ainda assim pode logar que ignorou uma duplicata);
   - Se **não estiver**, some o valor normalmente e adicione o `id` ao conjunto.
5. Simule uma falha no meio do lote (como nas seções anteriores) e "reinicie" o consumidor.

**Pergunta para o relatório:** mesmo com a mensagem sendo entregue duas vezes pelo Kafka (comportamento at-least-once), o total final bateu certinho com `15 * valor_fixo`? Por que essa solução funciona mesmo o Kafka não sabendo nada sobre o conceito de "id do pedido"?

**Desafio opcional:** o que aconteceria com essa solução se o consumidor inteiro fosse reiniciado (perdendo a variável `ids_processados` da memória)? Que mudança seria necessária para essa solução sobreviver a um reinício completo do processo (não seria mais em uma variável Python comum...)?


In [24]:
# Espaço para a Atividade 2

def consumir_idempotente(group_id, falhar_em=None, estado=None, offset_inicial=None):
    if estado is None:
        estado = {"total": 0.0, "ids_processados": set(), "processados": []}
    ids_processados = estado["ids_processados"]

    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        enable_auto_commit=False,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    tp = TopicPartition(TOPIC, 0)
    consumer.assign([tp])
    if offset_inicial is not None:
        consumer.seek(tp, offset_inicial)

    try:
        for msg in consumer:
            pedido = msg.value

            if pedido["id"] in ids_processados:
                print(f"Duplicata ignorada: sequência {pedido['sequencia']}")
            else:
                estado["total"] += pedido["valor"]
                ids_processados.add(pedido["id"])
                estado["processados"].append(pedido["sequencia"])

            # A falha vem depois do efeito, mas antes do commit: entrega at-least-once.
            if falhar_em is not None and pedido["sequencia"] == falhar_em:
                raise RuntimeError(f"Falha simulada na sequência {falhar_em}")

            consumer.commit()
    finally:
        consumer.close()

    return estado

inicio_lote_atividade2 = obter_offset_final(TOPIC)
publicar_pedidos(15)
grupo_atividade2 = f"atividade2-idempotente-{uuid.uuid4()}"
estado_atividade2 = {"total": 0.0, "ids_processados": set(), "processados": []}

try:
    consumir_idempotente(
        grupo_atividade2, falhar_em=7, estado=estado_atividade2,
        offset_inicial=inicio_lote_atividade2,
    )
except RuntimeError as erro:
    print(f"{erro}; reiniciando o consumidor...")

# O set é preservado para representar um reinício do consumidor no mesmo processo.
consumir_idempotente(grupo_atividade2, estado=estado_atividade2)
print(f"Total idempotente: {estado_atividade2['total']:.1f}")
print(f"Total esperado: {15 * 10.0:.1f}")
print(f"Quantidade de ids únicos processados: {len(estado_atividade2['ids_processados'])}")


/tmp/ipykernel_30852/1247799591.py:43: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  publicar_pedidos(15)
/tmp/ipykernel_30852/1247799591.py:8: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


15 pedidos publicados (valor total esperado: 150.0).
Falha simulada na sequência 7; reiniciando o consumidor...
Duplicata ignorada: sequência 7
Total idempotente: 150.0
Total esperado: 150.0
Quantidade de ids únicos processados: 15


**Resposta da Atividade 2.** Sim: o total final é **150,0**. O Kafka volta a entregar a mensagem cujo offset não foi commitado, mas a aplicação reconhece o `id` já presente em `ids_processados` e não repete a soma. Isso funciona porque a deduplicação é uma regra do consumidor; o Kafka não precisa interpretar o identificador do pedido. Se o processo inteiro reiniciasse, o conjunto em memória seria perdido e a duplicata voltaria a produzir efeito. Para sobreviver a isso, os IDs e o efeito do processamento deveriam ser persistidos de forma atômica em um armazenamento durável, por exemplo uma tabela de banco de dados com `id` como chave única e `INSERT ... ON CONFLICT DO NOTHING`.


### Atividade 3 — Comparando `read_committed` e `read_uncommitted` com uma transação abortada por engano

**Objetivo:** reforçar o comportamento de isolamento transacional visto na Seção 4, desta vez causando a falha você mesmo dentro do bloco de uma transação (não com um `abort_transaction()` já pronto).

**Tarefa:**
1. Crie um novo tópico chamado `pedidos-atividade3` (1 partição, fator de replicação 1).
2. Escreva uma função `publicar_lote_transacional(producer, pedidos, transactional_id, falhar_no_meio)` que:
   - Chama `init_transactions()` e `begin_transaction()`;
   - Envia os pedidos da lista `pedidos` um por um;
   - Se `falhar_no_meio=True`, chame `producer.flush()` e então lance uma exceção proposital (`raise RuntimeError(...)`) depois de enviar metade dos pedidos, mas **antes** de chamar `commit_transaction()`;
   - Use um bloco `try/except` para capturar essa exceção e chamar `abort_transaction()` dentro dele (simulando o que uma aplicação real faria ao detectar um erro no meio do processamento);
   - Se `falhar_no_meio=False`, envie todos os pedidos normalmente, chame `producer.flush()` e finalize com `commit_transaction()`.
3. Rode a função duas vezes: uma com `falhar_no_meio=True` (um lote de pedidos "A"), outra com `falhar_no_meio=False` (um lote de pedidos "B", com `transactional_id` diferente do primeiro).
4. Leia o tópico `pedidos-atividade3` duas vezes — uma com `isolation_level="read_committed"`, outra com `isolation_level="read_uncommitted"` — e compare quais pedidos aparecem em cada leitura.

**Pergunta para o relatório:** quantos pedidos o `read_committed` mostrou, e quantos o `read_uncommitted` mostrou? Isso confirma que o lote "A" (abortado) ficou completamente invisível para consumidores `read_committed`, mesmo que parte dele já tivesse sido fisicamente enviada ao broker antes do abort?


In [25]:
# Espaço para a Atividade 3

def publicar_lote_transacional(producer, pedidos, transactional_id, falhar_no_meio):
    if producer.config.get("transactional_id") != transactional_id:
        raise ValueError("O transactional_id informado deve ser o mesmo configurado no producer.")

    producer.init_transactions()
    producer.begin_transaction()
    metade = max(1, len(pedidos) // 2)

    try:
        for indice, pedido in enumerate(pedidos, start=1):
            producer.send(TOPIC_ATIVIDADE3, value=pedido)
            if falhar_no_meio and indice == metade:
                producer.flush()
                raise RuntimeError("Falha proposital no meio do lote")

        producer.flush()
        producer.commit_transaction()
        print(f"Lote {pedidos[0]['lote']} commitado ({len(pedidos)} pedidos).")
    except RuntimeError as erro:
        producer.abort_transaction()
        print(f"{erro}; lote {pedidos[0]['lote']} abortado.")

def ler_atividade3(isolation_level, offset_inicial):
    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        enable_auto_commit=False,
        isolation_level=isolation_level,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    tp = TopicPartition(TOPIC_ATIVIDADE3, 0)
    consumer.assign([tp])
    consumer.seek(tp, offset_inicial)
    mensagens = [msg.value for msg in consumer]
    consumer.close()
    return mensagens

TOPIC_ATIVIDADE3 = "pedidos-atividade3"
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(
        name=TOPIC_ATIVIDADE3, num_partitions=1, replication_factor=1
    )])
except TopicAlreadyExistsError:
    pass
finally:
    admin.close()

inicio_lote_atividade3 = obter_offset_final(TOPIC_ATIVIDADE3)
execucao = uuid.uuid4().hex[:8]
id_a = f"atividade3-A-{execucao}"
id_b = f"atividade3-B-{execucao}"
pedidos_a = [{"id": f"A-{i}", "lote": "A", "sequencia": i} for i in range(6)]
pedidos_b = [{"id": f"B-{i}", "lote": "B", "sequencia": i} for i in range(6)]

producer_a = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS, transactional_id=id_a,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
producer_b = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS, transactional_id=id_b,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

publicar_lote_transacional(producer_a, pedidos_a, id_a, falhar_no_meio=True)
publicar_lote_transacional(producer_b, pedidos_b, id_b, falhar_no_meio=False)
producer_a.close()
producer_b.close()

mensagens_committed = ler_atividade3("read_committed", inicio_lote_atividade3)
mensagens_uncommitted = ler_atividade3("read_uncommitted", inicio_lote_atividade3)
print(f"read_committed ({len(mensagens_committed)}): {mensagens_committed}")
print(f"read_uncommitted ({len(mensagens_uncommitted)}): {mensagens_uncommitted}")


Falha proposital no meio do lote; lote A abortado.
Lote B commitado (6 pedidos).


/tmp/ipykernel_30852/2489213698.py:26: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


read_committed (6): [{'id': 'B-0', 'lote': 'B', 'sequencia': 0}, {'id': 'B-1', 'lote': 'B', 'sequencia': 1}, {'id': 'B-2', 'lote': 'B', 'sequencia': 2}, {'id': 'B-3', 'lote': 'B', 'sequencia': 3}, {'id': 'B-4', 'lote': 'B', 'sequencia': 4}, {'id': 'B-5', 'lote': 'B', 'sequencia': 5}]
read_uncommitted (9): [{'id': 'A-0', 'lote': 'A', 'sequencia': 0}, {'id': 'A-1', 'lote': 'A', 'sequencia': 1}, {'id': 'A-2', 'lote': 'A', 'sequencia': 2}, {'id': 'B-0', 'lote': 'B', 'sequencia': 0}, {'id': 'B-1', 'lote': 'B', 'sequencia': 1}, {'id': 'B-2', 'lote': 'B', 'sequencia': 2}, {'id': 'B-3', 'lote': 'B', 'sequencia': 3}, {'id': 'B-4', 'lote': 'B', 'sequencia': 4}, {'id': 'B-5', 'lote': 'B', 'sequencia': 5}]


**Resposta da Atividade 3.** Com lotes de seis pedidos, o `read_committed` mostra **6 pedidos**, todos do lote B commitado. O `read_uncommitted` mostra **9 pedidos**: os seis do lote B e os três do lote A enviados antes da falha. Portanto, o experimento confirma que o lote A abortado fica completamente invisível para consumidores `read_committed`, embora metade dele tenha chegado fisicamente ao broker antes do abort.


## 6. Extra: as três semânticas têm o mesmo custo de desempenho?

Não. Cada uma tem uma origem diferente (e específica) de overhead — vale entender de onde vem cada custo, em vez de assumir apenas que "mais garantia = mais lento" de forma vaga.

### At-most-once — a mais rápida
- **Producer:** nenhum overhead adicional (nenhuma garantia extra sendo mantida).
- **Consumer:** o commit acontece imediatamente, sem esperar o processamento terminar — não há bloqueio adicional.
- O preço dessa velocidade é justamente a possibilidade de perder mensagens.

### At-least-once — praticamente no mesmo patamar
- O commit acontece depois do processamento, mas isso normalmente não é o gargalo — é só mais uma chamada de rede ao coordenador de grupo, geralmente rápida.
- **Onde pode pesar de verdade:** se o `commit()` for **síncrono** após *cada* mensagem individual (como fizemos nas Seções 2 e 3, por clareza didática), isso adiciona uma viagem de rede por mensagem — lento em produção. Aplicações reais costumam commitar em lote (a cada N mensagens, ou a cada X segundos), o que é bem mais rápido, mas aumenta a janela de mensagens que seriam reprocessadas em caso de falha.
- Ou seja, aqui a diferença de velocidade depende mais de **como o commit é implementado** do que da semântica em si.

### Exactly-once — a única com overhead estrutural de verdade
Duas frentes:

- **No producer:** a idempotência sozinha (`enable_idempotence=True`) tem overhead quase nulo — só adiciona um número de sequência a cada mensagem. Quem realmente custa são as **transações**: cada uma exige chamadas extras ao coordenador de transações (registrar quais partições participam, escrever o marcador de commit/abort ao final). Esse é um custo **fixo por transação**, não por mensagem — por isso, na prática, agrupa-se muitas mensagens em cada transação, em vez de abrir uma transação por mensagem, para diluir esse custo.
- **No consumer (`read_committed`):** um consumidor `read_committed` não pode entregar mensagens de uma transação ainda em aberto até ver o marcador de commit (ou abort) dela. Isso não reduz a taxa de processamento em si, mas aumenta a **latência de entrega** — o tempo entre a mensagem ser escrita no broker e ela ficar visível para esse consumidor.

| Semântica | Overhead principal | Onde dói |
|---|---|---|
| At-most-once | Nenhum | — |
| At-least-once | Rede do commit (se for síncrono por mensagem) | Configurável — normalmente irrelevante em produção |
| Exactly-once | Coordenação transacional + buffer no read_committed | Fixo por transação (producer) + latência de entrega (consumer) |

### Exercício opcional — medindo isso na prática

**Objetivo:** transformar essa tabela em números reais, medidos no seu próprio ambiente.

**Tarefa:**
1. Publique um lote de 200 pedidos (pode reaproveitar `publicar_pedidos(200)` da Seção 1, ou uma versão adaptada).
2. Cronometre (com `time.perf_counter()`, antes e depois) três cenários de consumo/produção diferentes sobre esse lote:
   - **(a)** commit síncrono após cada mensagem individual (o padrão at-least-once usado na Seção 3);
   - **(b)** commit em lote, uma vez a cada 20 mensagens (ou só uma vez ao final, depois de processar tudo);
   - **(c)** publicação das 200 mensagens dentro de uma única transação (`begin_transaction()` → 200 `send()` → `flush()` → `commit_transaction()`), comparando com publicar as mesmas 200 mensagens em 200 transações separadas (uma por mensagem).
3. Imprima o tempo total de cada cenário e calcule a razão entre eles (ex.: "o cenário (a) foi X vezes mais lento que o (b)").

**Pergunta para o relatório:** os números bateram com o que a tabela acima previa? Qual cenário teve a maior diferença relativa em relação ao esperado, e você consegue explicar por quê (dica: pense em quantas viagens de rede cada cenário realmente faz)?


In [26]:
# Espaço para o exercício opcional da Seção 6

import time

def medir_consumo(offset_inicial, commit_a_cada):
    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=f"desempenho-{commit_a_cada}-{uuid.uuid4()}",
        enable_auto_commit=False,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    )
    tp = TopicPartition(TOPIC, 0)
    consumer.assign([tp])
    consumer.seek(tp, offset_inicial)

    inicio = time.perf_counter()
    for quantidade, _ in enumerate(consumer, start=1):
        if quantidade % commit_a_cada == 0:
            consumer.commit()
        if quantidade == 200:
            break
    tempo = time.perf_counter() - inicio
    consumer.close()
    return tempo

# Os dois consumidores começam no mesmo offset e leem exatamente os mesmos 200 pedidos.
inicio_lote_desempenho = obter_offset_final(TOPIC)
publicar_pedidos(200)
tempo_a = medir_consumo(inicio_lote_desempenho, commit_a_cada=1)
tempo_b = medir_consumo(inicio_lote_desempenho, commit_a_cada=20)

# Cenário (c): uma transação grande contra 200 transações pequenas.
TOPIC_DESEMPENHO_TX = "pedidos-desempenho-tx"
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(
        name=TOPIC_DESEMPENHO_TX, num_partitions=1, replication_factor=1
    )])
except TopicAlreadyExistsError:
    pass
finally:
    admin.close()

def criar_produtor_tx(nome):
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        transactional_id=f"{nome}-{uuid.uuid4()}",
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )
    producer.init_transactions()
    return producer

producer_uma_tx = criar_produtor_tx("uma-transacao")
inicio = time.perf_counter()
producer_uma_tx.begin_transaction()
for i in range(200):
    producer_uma_tx.send(TOPIC_DESEMPENHO_TX, value={"sequencia": i})
producer_uma_tx.flush()
producer_uma_tx.commit_transaction()
tempo_c_uma_transacao = time.perf_counter() - inicio
producer_uma_tx.close()

producer_200_tx = criar_produtor_tx("duzentas-transacoes")
inicio = time.perf_counter()
for i in range(200):
    producer_200_tx.begin_transaction()
    producer_200_tx.send(TOPIC_DESEMPENHO_TX, value={"sequencia": i})
    producer_200_tx.flush()
    producer_200_tx.commit_transaction()
tempo_c_200_transacoes = time.perf_counter() - inicio
producer_200_tx.close()

print(f"(a) Commit por mensagem: {tempo_a:.4f} s")
print(f"(b) Commit a cada 20 mensagens: {tempo_b:.4f} s")
print(f"(a) foi {tempo_a / tempo_b:.2f} vezes mais lento que (b).")
print(f"(c) Uma transação com 200 mensagens: {tempo_c_uma_transacao:.4f} s")
print(f"(c) 200 transações de uma mensagem: {tempo_c_200_transacoes:.4f} s")
print(f"200 transações foram {tempo_c_200_transacoes / tempo_c_uma_transacao:.2f} vezes mais lentas.")


/tmp/ipykernel_30852/294776921.py:28: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  publicar_pedidos(200)
/tmp/ipykernel_30852/294776921.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


200 pedidos publicados (valor total esperado: 2000.0).


/tmp/ipykernel_30852/294776921.py:53: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer_uma_tx = criar_produtor_tx("uma-transacao")
/tmp/ipykernel_30852/294776921.py:63: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer_200_tx = criar_produtor_tx("duzentas-transacoes")


(a) Commit por mensagem: 0.4592 s
(b) Commit a cada 20 mensagens: 0.1413 s
(a) foi 3.25 vezes mais lento que (b).
(c) Uma transação com 200 mensagens: 0.0199 s
(c) 200 transações de uma mensagem: 16.3172 s
200 transações foram 819.72 vezes mais lentas.


**Resposta do exercício opcional.** Sim, os resultados confirmam a previsão da tabela: realizar um commit para cada mensagem custa mais que agrupar os commits, e abrir 200 transações custa muito mais que publicar as 200 mensagens em uma única transação. A maior diferença relativa ocorre na comparação transacional, porque cada transação individual exige sua própria coordenação e confirmação pela rede. Os tempos exatos dependem da máquina e aparecem na saída da célula.
